<a href="https://colab.research.google.com/github/GustavoHochgraf/training-eval-overlap/blob/master/notebooks/training_eval_overlap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Training-Eval Overlap Analysis: Carolina × PoetaV2

Quantify how much of the PoetaV2 evaluation benchmark data appears in the Carolina corpus (pre-training data for a Qwen-based Portuguese LLM).

**Method:** Semantic similarity search using BAAI/bge-m3 embeddings + FAISS.

**Runtime:** ~3-5 hours on Colab T4 (full corpus). Quick test with 1K docs: ~2 min.

## Cell 1: Setup & Install

In [1]:
!pip install -q sentence-transformers faiss-gpu datasets pandas tqdm matplotlib seaborn

ERROR: Could not find a version that satisfies the requirement faiss-gpu (from versions: none)
ERROR: No matching distribution found for faiss-gpu


In [2]:
import os
import re
import json
import unicodedata
from pathlib import Path
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import faiss
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# Optional: mount Google Drive for saving results
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = Path('/content/drive/MyDrive/overlap_results')
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Drive mounted. Results will be saved to {DRIVE_DIR}")
except Exception:
    DRIVE_DIR = Path('/content/results')
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Not on Colab or Drive not mounted. Results → {DRIVE_DIR}")

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

ModuleNotFoundError: No module named 'faiss'

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
# Set MAX_CAROLINA_DOCS to None for the full corpus, or an integer for a quick test.
MAX_CAROLINA_DOCS = 1000        # Quick test: 1000 docs (~2 min). Set to None for full run.
SIMILARITY_THRESHOLD = 0.85     # Cosine similarity threshold for "overlap"
TOP_K = 5                       # Nearest neighbors per query
EMBEDDING_MODEL = 'BAAI/bge-m3'
BATCH_SIZE = 64
MIN_DOC_LENGTH = 50             # Minimum character length for Carolina docs
BOOTSTRAP_N = 10_000
BOOTSTRAP_CI = 0.95

## Cell 2: Download Carolina Corpus

In [ ]:
print("Loading Carolina corpus from HuggingFace...")
carolina_ds = load_dataset("carolina-c4ai/corpus-carolina", split="train", streaming=True)

# Extract texts, filtering short/empty docs
carolina_texts = []
carolina_meta = []  # taxonomy info if available

for i, example in enumerate(tqdm(carolina_ds, desc="Loading Carolina")):
    if MAX_CAROLINA_DOCS is not None and i >= MAX_CAROLINA_DOCS:
        break

    text = example.get("text", "") or ""
    if len(text.strip()) < MIN_DOC_LENGTH:
        continue

    carolina_texts.append(text.strip())
    carolina_meta.append({
        "doc_id": str(i),
        "domain": example.get("domain", example.get("taxonomy", "unknown")),
        "length": len(text),
    })

print(f"\nCarolina corpus loaded: {len(carolina_texts):,} documents")
print(f"Avg length: {np.mean([m['length'] for m in carolina_meta]):.0f} chars")

# Taxonomy/domain distribution
domain_counts = pd.Series([m['domain'] for m in carolina_meta]).value_counts()
print(f"\nDomain distribution (top 10):")
print(domain_counts.head(10).to_string())

## Cell 3: Download & Extract PoetaV2 Tasks

In [ ]:
# Clone the PoetaV2 repository
!rm -rf /content/PoETaV2
!git clone --depth 1 https://github.com/PoETaV2/PoETaV2.git /content/PoETaV2
print("\nPoetaV2 repo cloned.")
!ls /content/PoETaV2/

In [ ]:
# Explore the repo structure to find task configs and dataset sources
import glob

poetav2_root = Path("/content/PoETaV2")

# Find all YAML/JSON config files that might define tasks
config_files = sorted(poetav2_root.rglob("*.yaml")) + sorted(poetav2_root.rglob("*.yml"))
print(f"Config files found: {len(config_files)}")
for f in config_files[:20]:
    print(f"  {f.relative_to(poetav2_root)}")

# Find Python files that define tasks/datasets
py_files = sorted(poetav2_root.rglob("*.py"))
print(f"\nPython files found: {len(py_files)}")
for f in py_files[:20]:
    print(f"  {f.relative_to(poetav2_root)}")

In [ ]:
# Parse task configurations to find HuggingFace dataset sources
import yaml

tasks_info = []

for config_path in sorted(poetav2_root.rglob("*.yaml")):
    try:
        with open(config_path) as f:
            config = yaml.safe_load(f)
        if config and isinstance(config, dict):
            # Look for dataset/task definitions
            task_name = config_path.stem
            dataset_name = config.get("dataset_path", config.get("dataset_name", config.get("dataset", None)))
            if dataset_name:
                tasks_info.append({
                    "task_name": task_name,
                    "dataset": dataset_name,
                    "subset": config.get("dataset_name", config.get("subset", None)),
                    "config_path": str(config_path.relative_to(poetav2_root)),
                })
    except Exception:
        pass

tasks_df = pd.DataFrame(tasks_info)
print(f"Found {len(tasks_df)} task configurations:")
if len(tasks_df) > 0:
    print(tasks_df.to_string(index=False))

In [ ]:
@dataclass
class EvalInstance:
    """A single evaluation instance from a PoetaV2 task."""
    task_name: str
    instance_id: str
    split: str = "test"
    question: str = ""
    context: str = ""
    answer: str = ""
    choices: list = field(default_factory=list)

    @property
    def search_text(self) -> str:
        """Concatenate question + context for embedding search."""
        parts = [self.question, self.context]
        return " ".join(p for p in parts if p.strip())

    @property
    def all_text_fields(self) -> list:
        fields = [self.question, self.context, self.answer] + self.choices
        return [f for f in fields if f.strip()]


def load_poetav2_tasks_from_hf(tasks_df):
    """Load PoetaV2 tasks from their HuggingFace dataset sources."""
    all_instances = []

    for _, row in tasks_df.iterrows():
        task_name = row['task_name']
        dataset_path = row['dataset']
        subset = row.get('subset')

        print(f"Loading {task_name} from {dataset_path}...", end=" ")
        try:
            kwargs = {"path": dataset_path}
            if subset and subset != dataset_path:
                kwargs["name"] = subset

            ds = load_dataset(**kwargs)

            # Try test split first, then validation
            for split_name in ["test", "validation", "train"]:
                if split_name in ds:
                    split_ds = ds[split_name]
                    break
            else:
                split_ds = list(ds.values())[0]
                split_name = list(ds.keys())[0]

            cols = split_ds.column_names
            for idx, example in enumerate(split_ds):
                inst = EvalInstance(
                    task_name=task_name,
                    instance_id=f"{task_name}_{idx}",
                    split=split_name,
                    question=str(example.get("question", example.get("input", example.get("sentence", "")))),
                    context=str(example.get("context", example.get("passage", example.get("premise", "")))),
                    answer=str(example.get("answer", example.get("target", example.get("output", example.get("label", ""))))),
                    choices=example.get("choices", example.get("options", [])),
                )
                if any(len(f) >= 20 for f in inst.all_text_fields):
                    all_instances.append(inst)

            print(f"✓ {len([i for i in all_instances if i.task_name == task_name])} instances")

        except Exception as e:
            print(f"✗ Error: {e}")

    return all_instances


def load_poetav2_tasks_from_jsonl(poetav2_dir):
    """Fallback: load tasks from JSONL files in the cloned repo."""
    all_instances = []
    for path in sorted(poetav2_dir.rglob("*.jsonl")):
        task_name = path.stem
        with open(path, encoding="utf-8") as fh:
            for idx, line in enumerate(fh):
                if not line.strip():
                    continue
                row = json.loads(line)
                inst = EvalInstance(
                    task_name=task_name,
                    instance_id=row.get("id", f"{task_name}_{idx}"),
                    split=row.get("split", "test"),
                    question=row.get("question", row.get("input", "")),
                    context=row.get("context", row.get("passage", "")),
                    answer=row.get("answer", row.get("target", row.get("output", ""))),
                    choices=row.get("choices", row.get("options", [])),
                )
                if any(len(f) >= 20 for f in inst.all_text_fields):
                    all_instances.append(inst)
    return all_instances

In [ ]:
# Load PoetaV2 instances — try HuggingFace first, fall back to JSONL
if len(tasks_df) > 0:
    print("Loading tasks from HuggingFace datasets...")
    eval_instances = load_poetav2_tasks_from_hf(tasks_df)
else:
    print("No HF dataset configs found. Loading from JSONL files...")
    eval_instances = []

# Also try JSONL files from the repo as fallback/supplement
jsonl_instances = load_poetav2_tasks_from_jsonl(poetav2_root)
if jsonl_instances:
    existing_tasks = {i.task_name for i in eval_instances}
    new_from_jsonl = [i for i in jsonl_instances if i.task_name not in existing_tasks]
    if new_from_jsonl:
        print(f"\nAdding {len(new_from_jsonl)} instances from JSONL (tasks not in HF)")
        eval_instances.extend(new_from_jsonl)

print(f"\nTotal: {len(eval_instances)} evaluation instances")

# Build DataFrame for display
instance_df = pd.DataFrame([
    {"task_name": i.task_name, "instance_id": i.instance_id,
     "question": i.question[:100], "context": i.context[:100],
     "answer": str(i.answer)[:50]}
    for i in eval_instances
])

# Per-task counts
task_counts = instance_df['task_name'].value_counts().sort_index()
print(f"\n{len(task_counts)} tasks found:")
print(task_counts.to_string())

## Cell 4: Embed Carolina with bge-m3

In [ ]:
print(f"Loading embedding model: {EMBEDDING_MODEL}")
model = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
embedding_dim = model.get_sentence_embedding_dimension()
print(f"Embedding dimension: {embedding_dim}")
print(f"Model loaded on {DEVICE}")

In [ ]:
# Encode Carolina documents in batches
# For large corpora, process in chunks to manage RAM

CHUNK_SIZE = 50_000  # Process this many docs at a time for RAM management

def build_faiss_index_incremental(texts, model, batch_size=64, chunk_size=50_000):
    """Build FAISS index incrementally to manage RAM."""
    dim = model.get_sentence_embedding_dimension()

    # Use GPU FAISS if available
    try:
        res = faiss.StandardGpuResources()
        index_flat = faiss.IndexFlatIP(dim)
        index = faiss.index_cpu_to_gpu(res, 0, index_flat)
        print("Using FAISS GPU index")
    except Exception:
        index = faiss.IndexFlatIP(dim)
        print("Using FAISS CPU index")

    n_chunks = (len(texts) + chunk_size - 1) // chunk_size

    for chunk_idx in range(n_chunks):
        start = chunk_idx * chunk_size
        end = min(start + chunk_size, len(texts))
        chunk_texts = texts[start:end]

        print(f"\nEncoding chunk {chunk_idx + 1}/{n_chunks} ({start:,}-{end:,})...")
        embeddings = model.encode(
            chunk_texts,
            batch_size=batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,
        )
        embeddings = np.array(embeddings, dtype=np.float32)
        index.add(embeddings)
        print(f"  Index size: {index.ntotal:,} vectors")

    return index


print(f"Encoding {len(carolina_texts):,} Carolina documents...")
carolina_index = build_faiss_index_incremental(
    carolina_texts, model, batch_size=BATCH_SIZE, chunk_size=CHUNK_SIZE
)
print(f"\nFAISS index built: {carolina_index.ntotal:,} vectors, dim={embedding_dim}")

In [ ]:
# Optional: save index to Drive
try:
    index_path = DRIVE_DIR / "faiss_index"
    index_path.mkdir(parents=True, exist_ok=True)

    # Convert GPU index to CPU for saving
    try:
        cpu_index = faiss.index_gpu_to_cpu(carolina_index)
    except Exception:
        cpu_index = carolina_index

    faiss.write_index(cpu_index, str(index_path / "carolina.faiss"))
    np.save(index_path / "doc_ids.npy", np.array([m['doc_id'] for m in carolina_meta], dtype=object))
    print(f"Index saved to {index_path}")
except Exception as e:
    print(f"Could not save index: {e}")

## Cell 5: Search for Overlap

In [ ]:
# Embed each PoetaV2 instance (question + context concatenated)
query_texts = [inst.search_text for inst in eval_instances]
print(f"Encoding {len(query_texts)} PoetaV2 queries...")

query_embeddings = model.encode(
    query_texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,
)
query_embeddings = np.array(query_embeddings, dtype=np.float32)
print(f"Query embeddings shape: {query_embeddings.shape}")

In [ ]:
# FAISS top-K nearest neighbor search
print(f"Searching FAISS index (top-{TOP_K})...")
scores, indices = carolina_index.search(query_embeddings, TOP_K)
print(f"Search complete. Scores shape: {scores.shape}")

# Build results DataFrame
results = []
for q_idx in range(len(eval_instances)):
    inst = eval_instances[q_idx]
    top1_sim = float(scores[q_idx][0])
    top1_doc_idx = int(indices[q_idx][0])

    top1_text = carolina_texts[top1_doc_idx] if top1_doc_idx >= 0 else ""
    top1_doc_id = carolina_meta[top1_doc_idx]['doc_id'] if top1_doc_idx >= 0 else ""

    results.append({
        'task_name': inst.task_name,
        'instance_id': inst.instance_id,
        'query_text': inst.search_text[:200],
        'top1_similarity': top1_sim,
        'top1_doc_id': top1_doc_id,
        'top1_doc_text': top1_text[:300],
        'overlap': top1_sim >= SIMILARITY_THRESHOLD,
        # Store all top-K similarities for analysis
        'top5_similarities': [float(scores[q_idx][k]) for k in range(TOP_K) if indices[q_idx][k] >= 0],
    })

results_df = pd.DataFrame(results)
print(f"\nResults: {len(results_df)} instances")
print(f"Overlapping (sim >= {SIMILARITY_THRESHOLD}): {results_df['overlap'].sum()} ({results_df['overlap'].mean()*100:.1f}%)")

## Cell 6: Results & Analysis

In [ ]:
# ── Per-task overlap table ─────────────────────────────────────────────────

def bootstrap_ci(contaminated, total, n_bootstrap=10_000, ci=0.95, seed=42):
    """Compute bootstrap confidence interval for a contamination rate."""
    if total == 0:
        return 0.0, 0.0, 0.0
    rng = np.random.default_rng(seed)
    labels = np.array([1] * contaminated + [0] * (total - contaminated))
    boot_rates = np.array([
        rng.choice(labels, size=total, replace=True).mean()
        for _ in range(n_bootstrap)
    ])
    alpha = (1 - ci) / 2
    return (
        contaminated / total,
        float(np.percentile(boot_rates, 100 * alpha)),
        float(np.percentile(boot_rates, 100 * (1 - alpha))),
    )


task_summary = []
for task_name, group in results_df.groupby('task_name'):
    total = len(group)
    overlapping = group['overlap'].sum()
    rate = overlapping / total if total > 0 else 0.0
    mean_sim = group['top1_similarity'].mean()
    _, ci_lo, ci_hi = bootstrap_ci(int(overlapping), total, BOOTSTRAP_N, BOOTSTRAP_CI)

    task_summary.append({
        'Task': task_name,
        'Total': total,
        'Overlapping': int(overlapping),
        'Rate (%)': rate * 100,
        'Mean Sim': mean_sim,
        'CI Lower': ci_lo * 100,
        'CI Upper': ci_hi * 100,
    })

summary_df = pd.DataFrame(task_summary).sort_values('Rate (%)', ascending=False)

# Overall row
total_all = results_df.shape[0]
overlap_all = results_df['overlap'].sum()
_, ci_lo_all, ci_hi_all = bootstrap_ci(int(overlap_all), total_all, BOOTSTRAP_N, BOOTSTRAP_CI)
overall_row = pd.DataFrame([{
    'Task': 'OVERALL',
    'Total': total_all,
    'Overlapping': int(overlap_all),
    'Rate (%)': overlap_all / total_all * 100 if total_all else 0,
    'Mean Sim': results_df['top1_similarity'].mean(),
    'CI Lower': ci_lo_all * 100,
    'CI Upper': ci_hi_all * 100,
}])
display_df = pd.concat([summary_df, overall_row], ignore_index=True)

print("Per-Task Overlap Summary")
print("=" * 90)
print(display_df.to_string(index=False, float_format='%.2f'))

In [ ]:
# ── Similarity distribution histogram ──────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of all top-1 similarities
ax = axes[0]
ax.hist(results_df['top1_similarity'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
ax.axvline(SIMILARITY_THRESHOLD, color='red', linestyle='--', linewidth=2, label=f'Threshold = {SIMILARITY_THRESHOLD}')
ax.set_xlabel('Top-1 Cosine Similarity')
ax.set_ylabel('Count')
ax.set_title('Distribution of Top-1 Similarity Scores')
ax.legend()

# Overlap rate per task (bar chart)
ax = axes[1]
plot_df = summary_df.sort_values('Rate (%)', ascending=True)
colors = ['#e74c3c' if r > 0 else '#2ecc71' for r in plot_df['Rate (%)']]
ax.barh(plot_df['Task'], plot_df['Rate (%)'], color=colors, edgecolor='black', alpha=0.8)
ax.set_xlabel('Overlap Rate (%)')
ax.set_title('Overlap Rate per Task')
ax.set_xlim(0, max(plot_df['Rate (%)'].max() * 1.1, 1))

plt.tight_layout()
plt.savefig(DRIVE_DIR / 'overlap_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"Figure saved to {DRIVE_DIR / 'overlap_distribution.png'}")

In [ ]:
# ── Heatmap: overlap rate per task ─────────────────────────────────────────

if len(summary_df) > 1:
    fig, ax = plt.subplots(figsize=(4, max(6, len(summary_df) * 0.35)))
    heatmap_data = summary_df.set_index('Task')[['Rate (%)']]
    sns.heatmap(
        heatmap_data, annot=True, fmt='.1f', cmap='YlOrRd',
        cbar_kws={'label': 'Overlap Rate (%)'},
        linewidths=0.5, ax=ax
    )
    ax.set_title('Semantic Overlap Rate per Task')
    plt.tight_layout()
    plt.savefig(DRIVE_DIR / 'overlap_heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Heatmap saved to {DRIVE_DIR / 'overlap_heatmap.png'}")

In [ ]:
# ── Top-N most suspicious matches (manual inspection) ──────────────────────

TOP_N_INSPECT = 20

suspicious = results_df.nlargest(TOP_N_INSPECT, 'top1_similarity')
print(f"Top-{TOP_N_INSPECT} most similar matches (for manual inspection):")
print("=" * 100)
for _, row in suspicious.iterrows():
    print(f"\n{'─' * 80}")
    print(f"Task: {row['task_name']} | Instance: {row['instance_id']} | Similarity: {row['top1_similarity']:.4f}")
    print(f"Query:    {row['query_text'][:200]}")
    print(f"Matched:  {row['top1_doc_text'][:200]}")

In [ ]:
# ── Export results to CSV ──────────────────────────────────────────────────

# Full results
export_df = results_df.drop(columns=['top5_similarities'])
csv_path = DRIVE_DIR / 'overlap_results_full.csv'
export_df.to_csv(csv_path, index=False)
print(f"Full results saved to {csv_path}")

# Summary table
summary_csv_path = DRIVE_DIR / 'overlap_summary_per_task.csv'
display_df.to_csv(summary_csv_path, index=False)
print(f"Summary table saved to {summary_csv_path}")

# Download links (Colab)
try:
    from google.colab import files
    files.download(str(csv_path))
    files.download(str(summary_csv_path))
except Exception:
    pass

## Cell 7: LaTeX Table

In [ ]:
# ── Generate publication-ready LaTeX table with bootstrap CIs ──────────────

def generate_latex_table(summary_df, results_df, n_bootstrap=10_000, ci=0.95):
    """Generate a LaTeX table with per-task overlap rates and bootstrap CIs."""
    lines = [
        r"\begin{table}[ht]",
        r"\centering",
        r"\caption{Semantic overlap rates: Carolina corpus $\times$ PoetaV2 benchmarks.}",
        r"\label{tab:semantic-overlap}",
        r"\begin{tabular}{lrrrr}",
        r"\toprule",
        r"Task & $N$ & Overlap (\%) & Mean Sim & 95\% CI \\",
        r"\midrule",
    ]

    for _, row in summary_df.iterrows():
        task = row['Task'].replace('_', r'\_')
        lines.append(
            f"{task} & {row['Total']:.0f} & "
            f"{row['Rate (%)']:.1f} & {row['Mean Sim']:.3f} & "
            f"[{row['CI Lower']:.1f}, {row['CI Upper']:.1f}] \\\\"
        )

    # Overall row
    total_n = results_df.shape[0]
    total_overlap = results_df['overlap'].sum()
    rate = total_overlap / total_n * 100 if total_n else 0
    mean_sim = results_df['top1_similarity'].mean()
    _, ci_lo, ci_hi = bootstrap_ci(int(total_overlap), total_n, n_bootstrap, ci)

    lines.append(r"\midrule")
    lines.append(
        f"\\textbf{{Overall}} & {total_n} & "
        f"{rate:.1f} & {mean_sim:.3f} & "
        f"[{ci_lo * 100:.1f}, {ci_hi * 100:.1f}] \\\\"
    )

    lines += [
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{table}",
    ]

    return "\n".join(lines)


latex_table = generate_latex_table(summary_df, results_df, BOOTSTRAP_N, BOOTSTRAP_CI)

# Save to file
latex_path = DRIVE_DIR / 'overlap_table.tex'
latex_path.write_text(latex_table, encoding='utf-8')
print(f"LaTeX table saved to {latex_path}")

# Display for copy-paste
print("\n" + "=" * 80)
print("Copy-paste the following into your paper:")
print("=" * 80)
print(latex_table)